In [0]:
customers_schema = 'member_id string, emp_title string,emp_length string,home_ownership string, annual_inc float,addr_state string, zip_code string, country string, grade string,sub_grade string, verification_status string,tot_hi_cred_limit float, application_type string,annual_inc_joint float, verification_status_joint string'

In [0]:
customers_raw_df = spark.read \
    .format("csv") \
    .option("header","true") \
    .schema(customers_schema) \
    .load("/Volumes/finance_domain_project/default/raw_data/Raw/customers_data_csv")

In [0]:
customers_df_renamed = customers_raw_df.withColumnRenamed("annual_inc","annual_income") \
.withColumnRenamed("addr_state","address_state") \
.withColumnRenamed("zip_code","address_zip_code") \
.withColumnRenamed("country","address_country") \
.withColumnRenamed("tot_hi_cred_lim","total_high_credit_limit") \
.withColumnRenamed("annual_inc_joint","join_annual_income")


In [0]:
from pyspark.sql.functions import current_timestamp

In [0]:
customers_df_ingested=customers_df_renamed.withColumn("ingest_date",current_timestamp())

In [0]:
customers_df_ingested.count()

In [0]:
customers_distinct = customers_df_ingested.distinct()

In [0]:
customers_distinct.count()

In [0]:
customers_distinct.createOrReplaceTempView("customers")

In [0]:
display(spark.sql("select count(*) from customers where annual_income is null"))

In [0]:
customers_income_filtered = spark.sql("select * from customers where annual_income is not null")

In [0]:
customers_income_filtered.createOrReplaceTempView("customers")

In [0]:
spark.sql("select distinct(emp_length) from customers").show()

In [0]:
from pyspark.sql.functions import regexp_replace, col

In [0]:
customers_emp_length_cleaned= customers_income_filtered.withColumn("emp_length",regexp_replace(col("emp_length"),"(\D)",""))

In [0]:
customers_emp_length_casted = customers_emp_length_cleaned.withColumn("emp_length",customers_emp_length_cleaned.emp_length.cast('int'))

In [0]:
customers_emp_length_casted.filter("emp_length is null").count()

In [0]:
customers_emp_length_casted.createOrReplaceTempView("customers")


In [0]:
avg_emp_length=spark.sql("select floor(avg(emp_length)) as avg_emp_length from customers").collect()

In [0]:
print(avg_emp_length)

In [0]:
avg_emp_length[0][0]

In [0]:
avg_emp_duration= avg_emp_length[0][0]

In [0]:
print(avg_emp_duration)

In [0]:
customers_emplength_replaced = customers_emp_length_casted.na.fill(avg_emp_duration,subset=['emp_length'])

In [0]:
customers_emplength_replaced.filter("emp_length is null").count()

In [0]:
customers_emplength_replaced.createOrReplaceTempView("customers")

In [0]:
display(spark.sql("select count(address_state) from customers where length(address_state)>2"))

In [0]:
from pyspark.sql.functions import when, col, length

In [0]:
customers_state_cleaned=customers_emplength_replaced.withColumn("address_state",when(length(col("address_state"))>2,"NA").otherwise(col("address_state")))

In [0]:
display(customers_state_cleaned.select("address_state").distinct())

In [0]:
dbutils.fs.mkdirs("/Volumes/finance_domain_project/default/raw_data/Cleaned")



In [0]:
customers_state_cleaned.write\
.format("parquet") \
.mode("overwrite") \
.option("path","/Volumes/finance_domain_project/default/raw_data/Cleaned/customers_parquet") \
.save()


In [0]:
customers_state_cleaned.write\
.option("header",True) \
.format("csv") \
.mode("overwrite") \
.option("path","/Volumes/finance_domain_project/default/raw_data/Cleaned/customers_csv") \
.save()
